# M3 experiments: inventory, checkpoint verification, write test and clean-reproduction check
Run order: **inventory** → m3_build_verify → train_m3 → corruption → analysis

Run on: Colab **T4 GPU**. The saved Nepal results were produced on a T4, so section 6 can test for an *exact* match; on a
CPU runtime section 6 runs without mixed precision and small differences are expected. About 15-25 min, under 1 compute unit.

**What this notebook writes:** only inside `MAAI 902 X 901 PROJECT/m3_experiments/` (every Drive path it writes to goes
through `out()`, which refuses anything outside that folder). It **reads** `checkpoints/`, `eval_extras/nepal_final/`
and `xbd/` and changes nothing there. Section 7 re-checks that nothing outside `m3_experiments/` changed.

**What it does not do:** no M3 code, no training, no corruption. Section 6 re-predicts 64 clean Nepal crops with 4
existing checkpoints only to check that the pipeline reproduces the saved Nepal evaluation.

Sections:
0. Setup and a read-only snapshot of the existing project files
1. Create `m3_experiments/` and test write access
2. Code pinned to `main@8f10e61`; equivalence with the branch the Nepal run used; environment versions
3. Physical verification of the 12 checkpoints (files, hash, strict load, config, best epoch, forward pass)
4. M1 weighted-CE class weights vs the values proposed for M3
5. The saved Nepal results: presence, crop set and order, metrics recomputed from the saved matrices
6. Clean-reproduction spot check: 64 Nepal crops re-predicted and compared crop by crop with the saved results
7. Preservation check and summary

## 0. Setup

In [ ]:
# Which GPU this session has (the saved Nepal results came from a T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "no GPU (CPU runtime)"
import os, sys, json, time, hashlib, datetime, platform, shutil, math, subprocess
from google.colab import drive
drive.mount("/content/drive")

# Project paths on Drive and on the local Colab disk
D = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"      # same project root as every existing notebook
CK = f"{D}/checkpoints"
EXTRAS = f"{D}/eval_extras"
NEPAL = f"{EXTRAS}/nepal_final"
EXP = f"{D}/m3_experiments"                               # the ONLY Drive folder this notebook writes to
X, MAN = "/content/xbd", "/content/xbd/rscc_flood_manifest.csv"   # local Colab disk, as in the existing notebooks
REPO, REPO_COMMIT = "/content/FloodVerify", "8f10e61"
M3_CLASS_WEIGHTS = [0.10, 0.52, 0.88, 0.75, 2.74]         # proposed for M3 (= the M1 weighted-CE values, checked in 4)

# Stop early if an existing project folder is missing
for p in (D, CK, EXTRAS, NEPAL, f"{D}/xbd"):
    assert os.path.isdir(p), f"MISSING: {p}"
print("project folder found. Top level:", sorted(os.listdir(D)))
print("m3_experiments already exists:", os.path.isdir(EXP))

# out(): every path this notebook writes to is built here; anything outside m3_experiments/ raises
_EXP_REAL = os.path.realpath(EXP)
def out(*parts):
    # Path inside m3_experiments; raises for anything that would land elsewhere.
    p = os.path.realpath(os.path.join(EXP, *parts))
    if not (p == _EXP_REAL or p.startswith(_EXP_REAL + os.sep)):
        raise ValueError(f"refusing to write outside m3_experiments: {p}")
    return p

# Write JSON, converting numpy values to plain Python numbers
def jdump(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))

def snapshot():
    # Read-only: size and modification time of every file the existing work relies on.
    snap = {}
    for name in sorted(os.listdir(D)):
        if name != "m3_experiments":
            snap[f"[top] {name}"] = os.path.getmtime(f"{D}/{name}")
    for root in (CK, EXTRAS, f"{D}/figures"):
        if os.path.isdir(root):
            for dirpath, _, files in os.walk(root):
                for fn in files:
                    st = os.stat(os.path.join(dirpath, fn))
                    snap[os.path.relpath(os.path.join(dirpath, fn), D)] = [st.st_size, st.st_mtime]
    ev = f"{D}/evidence_predicted"
    if os.path.isdir(ev):
        for sub in sorted(os.listdir(ev)):
            snap[f"evidence_predicted/{sub} (file count)"] = len(os.listdir(f"{ev}/{sub}"))
    return snap

# Snapshot before any work; compared with a second snapshot in section 7
t0 = time.time(); BEFORE = snapshot()
print(f"read-only snapshot: {len(BEFORE)} entries in {time.time() - t0:.0f}s")

name, memory.total [MiB]
Tesla T4, 15360 MiB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
project folder found. Top level: ['SEM 1 FINAL PROJECT', 'advisory_generation', 'checkpoints', 'eval_extras', 'evidence_predicted', 'figures', 'guidance', 'humaid', 'xbd', 'xbd_tars']
m3_experiments already exists: False
read-only snapshot: 120 entries in 20s


## 1. Create `m3_experiments/` and test write access

In [ ]:
# Folder layout inside m3_experiments/
SUBDIRS = ["code", "notebooks", "configs", "inventory", "inventory/clean_spotcheck",
           "m3/checkpoints", "m3/benchmark", "m3/eval_val",
           "corruption/manifest", "corruption/input_hashes", "corruption/samples", "corruption/predictions",
           "results", "figures", "logs"]
try:
    for s in SUBDIRS:
        os.makedirs(out(s), exist_ok=True)
except OSError as e:
    raise RuntimeError(f"Cannot create folders inside the shared project folder ({e!r}). This account probably has "
                       "view-only access: ask the folder owner for Editor access.") from e

# Write test: write a random token, read it back, then delete the probe file
WT = {"time": datetime.datetime.now().isoformat(), "token": hashlib.sha256(os.urandom(16)).hexdigest()[:16]}
probe = out("inventory", "_write_probe.tmp")
try:
    with open(probe, "w") as f:
        f.write(WT["token"]); f.flush(); os.fsync(f.fileno())
    WT["write"] = True
    WT["read_back_matches"] = open(probe).read() == WT["token"]
    os.remove(probe)
    WT["delete_own_probe_file"] = not os.path.exists(probe)
except OSError as e:
    WT["error"] = repr(e)
WT["verdict"] = ("WRITE ACCESS OK" if WT.get("write") and WT.get("read_back_matches") and WT.get("delete_own_probe_file")
                 else "WRITE ACCESS FAILED")
print(json.dumps(WT, indent=2))
assert WT["verdict"] == "WRITE ACCESS OK", "stop: fix Drive permissions first"
jdump(WT, out("inventory", "write_test.json"))

# README for the new folder (written once, never overwritten)
if not os.path.exists(out("README.md")):
    open(out("README.md"), "w").write(
        "# m3_experiments\n\n"
        "New work only: M3 (multi-scale temporal attention U-Net), its two-head ablation, and the controlled\n"
        "post-image corruption study on Nepal. Nothing outside this folder is written by these notebooks.\n"
        "Existing models (M0-M2) are READ from ../checkpoints/; saved Nepal results are READ from\n"
        "../eval_extras/nepal_final/. Code pinned to github.com/aaarrvind/FloodVerify @ 8f10e61.\n\n"
        "Folders: code/ notebooks/ configs/ inventory/ m3/{checkpoints,benchmark,eval_val}/\n"
        "corruption/{manifest,input_hashes,samples,predictions}/ results/ figures/ logs/\n")
print(sorted(os.listdir(EXP)))

{
  "time": "2026-11-20T11:59:18.442441",
  "token": "58a3f0aa7e780bef",
  "write": true,
  "read_back_matches": true,
  "delete_own_probe_file": true,
  "verdict": "WRITE ACCESS OK"
}
['README.md', 'code', 'configs', 'corruption', 'figures', 'inventory', 'logs', 'm3', 'notebooks', 'results']


## 2. Code pinned to `main@8f10e61`
The Nepal notebook cloned branch `m3/eval-extras`. This cell checks that every file affecting preprocessing, models,
inference and metrics is identical between that branch and the pinned commit, and records library versions
(a newer `transformers` or `torch` on Colab is a possible source of differences).

In [ ]:
# Fresh clone of the repository, pinned to commit 8f10e61 and installed in editable mode
%cd /content
!rm -rf /content/FloodVerify
!git clone -q https://github.com/aaarrvind/FloodVerify.git /content/FloodVerify
%cd /content/FloodVerify
!git checkout -q 8f10e61 && git log -1 --format='%h %cd %s'
!pip install -q -e . --no-deps jsonschema
# Make the repo's package importable in this kernel
sys.path.insert(0, "/content/FloodVerify/src")

/content
/content/FloodVerify
8f10e61 Fri Nov 20 14:07:06 2026 +0400 Merge pull request #28 from aaarrvind/m5/generate-advisory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for floodsitrep (pyproject.toml) ... done


In [ ]:
# Run a git command in the cloned repo and return its output
def git(*a):
    return subprocess.run(["git", "-C", REPO, *a], capture_output=True, text=True).stdout.strip()

# Files that affect preprocessing, models, inference or metrics
RELEVANT = ["src/floodsitrep/data/dataset.py", "src/floodsitrep/data/tiling.py", "src/floodsitrep/data/splits.py",
            "src/floodsitrep/data/labels_to_evidence.py", "src/floodsitrep/cv/eval/infer.py",
            "src/floodsitrep/cv/eval/building_level.py", "src/floodsitrep/cv/eval/metrics.py",
            "src/floodsitrep/cv/eval/extras.py", "src/floodsitrep/cv/models/__init__.py",
            "src/floodsitrep/cv/models/siamese_unet.py", "src/floodsitrep/cv/models/siamese_segformer.py",
            "src/floodsitrep/cv/models/losses.py", "src/floodsitrep/cv/train.py"]
# Compare those files between the branch used for the Nepal run and the pinned commit
branch_head = git("log", "-1", "--format=%h %cI", "origin/m3/eval-extras")
diff = git("diff", "--name-only", "origin/m3/eval-extras", REPO_COMMIT, "--", *RELEVANT)
# When the Nepal decision file was written (the freeze time of the Nepal evaluation)
dec_mtime = (datetime.datetime.fromtimestamp(os.path.getmtime(f"{NEPAL}/decision.txt")).isoformat()
             if os.path.exists(f"{NEPAL}/decision.txt") else "decision.txt MISSING")
# Library versions in this session
import torch, torchvision, numpy, scipy, PIL, transformers, pandas
ENV = {"torch": torch.__version__, "torchvision": torchvision.__version__, "transformers": transformers.__version__,
       "numpy": numpy.__version__, "scipy": scipy.__version__, "pillow": PIL.__version__, "pandas": pandas.__version__,
       "python": platform.python_version(),
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)"}
CODE_EQ = {"pinned_commit": git("log", "-1", "--format=%h %cI %s"), "nepal_run_branch_head": branch_head,
           "nepal_decision_txt_written": dec_mtime, "relevant_files": RELEVANT,
           "files_differing": diff.splitlines(), "identical": diff == "", "environment": ENV}
jdump(CODE_EQ, out("inventory", "code_equivalence.json"))
print(json.dumps(CODE_EQ, indent=2))

{
  "pinned_commit": "8f10e61 2026-11-20T14:07:06+04:00 Merge pull request #28 from aaarrvind/m5/generate-advisory",
  "nepal_run_branch_head": "4624338 2026-11-17T16:48:36+04:00",
  "nepal_decision_txt_written": "2026-11-18T13:29:49",
  "relevant_files": [
    "src/floodsitrep/data/dataset.py",
    "src/floodsitrep/data/tiling.py",
    "src/floodsitrep/data/splits.py",
    "src/floodsitrep/data/labels_to_evidence.py",
    "src/floodsitrep/cv/eval/infer.py",
    "src/floodsitrep/cv/eval/building_level.py",
    "src/floodsitrep/cv/eval/metrics.py",
    "src/floodsitrep/cv/eval/extras.py",
    "src/floodsitrep/cv/models/__init__.py",
    "src/floodsitrep/cv/models/siamese_unet.py",
    "src/floodsitrep/cv/models/siamese_segformer.py",
    "src/floodsitrep/cv/models/losses.py",
    "src/floodsitrep/cv/train.py"
  ],
  "files_differing": [],
  "identical": true,
  "environment": {
    "torch": "2.14.1+cu130",
    "torchvision": "0.29.1+cu130",
    "transformers": "5.22.1",
    "numpy": "2.

## 3. Physical verification of the 12 checkpoints
For every run: files present, `best.pt` size and SHA-256, `torch.load`, the stored config against the expected recipe
(model, loss, class weights, lr, encoder lr multiplier, weight decay, warmup, epochs, batch size, seed, run name),
`config.yaml` against the stored config, the best epoch against `metrics.jsonl`, a **strict** load through the repo's own
`load_checkpoint` (the loader every evaluation uses), parameter count, and a forward pass on a fixed random input.
Any problem is recorded, not hidden: status `OK`, `CHECK` (loads, but something differs) or `FAILED`.

In [ ]:
import numpy as np, pandas as pd, yaml
from floodsitrep.cv.models import load_checkpoint

# Expected training recipe of each existing model family
FAMILIES = {   # expected recipe per family (from configs/cv/*.yaml and the --set overrides in the training notebooks)
    "M0 Dice+focal":   dict(prefix="m0_early_fusion_unet", model="early_fusion_unet", loss="dice_focal",
                            lr=3e-4, encoder_lr_mult=1.0, weight_decay=1e-4, warmup_steps=200),
    "M1 weighted CE":  dict(prefix="m1_siamese_unet_ce", model="siamese_unet", loss="ce",
                            lr=3e-4, encoder_lr_mult=1.0, weight_decay=1e-4, warmup_steps=200),
    "M1 Dice+focal":   dict(prefix="m1_siamese_unet", model="siamese_unet", loss="dice_focal",
                            lr=3e-4, encoder_lr_mult=1.0, weight_decay=1e-4, warmup_steps=200),
    "M2 SegFormer CE": dict(prefix="m2_siamese_segformer_ce", model="siamese_segformer", loss="ce",
                            lr=6e-4, encoder_lr_mult=0.1, weight_decay=1e-2, warmup_steps=500,
                            backbone="nvidia/mit-b1"),
}
# Settings shared by all four families
COMMON = {"epochs": 20, "batch_size": 8, "class_weights": M3_CLASS_WEIGHTS, "n_classes": 5, "amp": True}
# The 12 expected run folders: 4 families x seeds 0-2
EXPECTED = {f"{f['prefix']}_s{s}": (fam, s) for fam, f in FAMILIES.items() for s in (0, 1, 2)}

# Which run folders exist on Drive
on_drive = sorted(d for d in os.listdir(CK) if os.path.isdir(f"{CK}/{d}"))
print("run folders on Drive:", len(on_drive))
print("in this study (12 expected):", sum(r in on_drive for r in EXPECTED))
print("MISSING run folders:", [r for r in EXPECTED if r not in on_drive] or "none")
print("on Drive but not in this study:", [d for d in on_drive if d not in EXPECTED])

# SHA-256 of a file, read in 4 MB chunks
def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

# List every stored config value that differs from the expected recipe
def config_issues(cfg, fam, seed, run):
    exp = {**COMMON, **{k: v for k, v in FAMILIES[fam].items() if k != "prefix"}, "seed": seed, "run_name": run}
    issues = []
    for k, v in exp.items():
        got = cfg.get(k, "<absent>")
        if isinstance(v, float):
            ok = isinstance(got, (int, float)) and not isinstance(got, bool) and math.isclose(got, v, rel_tol=1e-9)
        else:
            ok = got == v
        if not ok:
            issues.append(f"{k}: expected {v!r}, got {got!r}")
    return issues

# Fixed random input for the forward-pass check
torch.manual_seed(0)
XIN = torch.randn(1, 3, 128, 128)
INV, DETAILS = [], {}
# Check each run; problems are recorded, not raised, so every run is reported
for run, (fam, seed) in EXPECTED.items():
    r, d, issues = {"run": run, "family": fam, "seed": seed}, f"{CK}/{run}", []
    t0 = time.time()
    try:
        if not os.path.isdir(d):
            raise FileNotFoundError("run folder missing")
        files = {fn: os.path.getsize(f"{d}/{fn}") for fn in os.listdir(d)}
        r["files"] = " ".join(sorted(files))
        for fn in ("best.pt", "last.pt", "config.yaml", "metrics.jsonl"):
            r[f"has_{fn}"] = fn in files
        if not r["has_best.pt"]:
            raise FileNotFoundError("best.pt missing")
        r["best_pt_MB"] = round(files["best.pt"] / 1e6, 1)
        r["best_pt_modified"] = datetime.datetime.fromtimestamp(os.path.getmtime(f"{d}/best.pt")).isoformat(timespec="minutes")
        r["best_pt_sha256"] = sha256(f"{d}/best.pt")
        # Load the checkpoint and compare its stored config with the expected recipe
        ck = torch.load(f"{d}/best.pt", map_location="cpu", weights_only=False)
        r["ckpt_keys"] = " ".join(sorted(ck))
        cfg = ck["config"]
        r["best_epoch"] = int(ck["epoch"])
        r["ckpt_val_score"] = float(ck["val"]["xview2_pixel_score"])
        issues += config_issues(cfg, fam, seed, run)
        if r["has_config.yaml"]:
            y = yaml.safe_load(open(f"{d}/config.yaml"))
            dif = [k for k in y if k != "encoder_config" and y[k] != cfg.get(k)]
            if dif:
                issues.append("config.yaml differs from the checkpoint's config on: " + ", ".join(dif))
        else:
            issues.append("no config.yaml")
        # The best epoch in the training log must match the epoch saved in best.pt
        if r["has_metrics.jsonl"]:
            m = pd.DataFrame([json.loads(l) for l in open(f"{d}/metrics.jsonl")])
            r["epochs_logged"], r["distinct_epochs"] = len(m), m["epoch"].nunique()
            bi = m["xview2_pixel_score"].idxmax()
            r["log_best_epoch"], r["log_best_score"] = int(m.loc[bi, "epoch"]), float(m.loc[bi, "xview2_pixel_score"])
            if r["log_best_epoch"] != r["best_epoch"] or not math.isclose(r["log_best_score"], r["ckpt_val_score"], abs_tol=1e-9):
                issues.append(f"best.pt is epoch {r['best_epoch']} ({r['ckpt_val_score']:.4f}) but the log's best is "
                              f"epoch {r['log_best_epoch']} ({r['log_best_score']:.4f})")
            if r["distinct_epochs"] != cfg.get("epochs") or r["epochs_logged"] != r["distinct_epochs"]:
                issues.append(f"metrics.jsonl: {r['epochs_logged']} lines, {r['distinct_epochs']} distinct epochs, "
                              f"config says {cfg.get('epochs')}")
        else:
            issues.append("no metrics.jsonl")
        # Strict load through the repo's own loader, then a forward pass
        model, _ = load_checkpoint(f"{d}/best.pt", "cpu")       # strict load_state_dict inside
        r["params_M"] = round(sum(p.numel() for p in model.parameters()) / 1e6, 3)
        with torch.no_grad():
            yout = model(XIN, XIN)
        r["forward_shape"] = str(tuple(yout.shape))
        if tuple(yout.shape) != (1, 5, 128, 128) or not torch.isfinite(yout).all():
            issues.append(f"forward output {tuple(yout.shape)}, finite={bool(torch.isfinite(yout).all())}")
        DETAILS[run] = {"config": {k: v for k, v in cfg.items() if k != "encoder_config"},
                        "encoder_config_stored": "encoder_config" in cfg, "issues": issues}
        r["status"] = "OK" if not issues else "CHECK"
        del model, ck
    except Exception as e:
        r["status"], issues = "FAILED", issues + [repr(e)]
    r["issues"] = " | ".join(issues)
    r["seconds"] = round(time.time() - t0, 1)
    INV.append(r)
    print(f"{run:30s} {r['status']:6s} epoch {r.get('best_epoch', '-'):>2}  val {r.get('ckpt_val_score', float('nan')):.4f}  "
          f"{r.get('params_M', '-')} M  {r['seconds']}s  {r['issues']}", flush=True)

# Save the inventory table and the stored configs
INV_DF = pd.DataFrame(INV)
INV_DF.to_csv(out("inventory", "checkpoint_inventory.csv"), index=False)
jdump(DETAILS, out("inventory", "checkpoint_configs.json"))

# runs on Drive that are not in the study: file presence only (no loading)
OTHER = [{"run": dd, "files": " ".join(sorted(os.listdir(f"{CK}/{dd}")))} for dd in on_drive if dd not in EXPECTED]
pd.DataFrame(OTHER).to_csv(out("inventory", "other_run_folders.csv"), index=False)
print("\nstatus counts:", INV_DF.status.value_counts().to_dict())

run folders on Drive: 16
in this study (12 expected): 12
MISSING run folders: none
on Drive but not in this study: ['m2_siamese_segformer_ade_ce_s0', 'm2_siamese_segformer_gentle_ce_s0', 'smoke_m1_s0', 'smoke_m2']
m0_early_fusion_unet_s0        OK     epoch  9  val 0.2858  24.446 M  8.3s  
m0_early_fusion_unet_s1        OK     epoch  7  val 0.2738  24.446 M  7.2s  
m0_early_fusion_unet_s2        OK     epoch  8  val 0.2677  24.446 M  10.8s  
m1_siamese_unet_ce_s0          OK     epoch 12  val 0.3633  25.496 M  9.2s  
m1_siamese_unet_ce_s1          OK     epoch 12  val 0.3626  25.496 M  9.3s  
m1_siamese_unet_ce_s2          OK     epoch 18  val 0.3561  25.496 M  6.1s  
m1_siamese_unet_s0             OK     epoch  9  val 0.3157  25.496 M  8.7s  
m1_siamese_unet_s1             OK     epoch 14  val 0.3258  25.496 M  9.2s  
m1_siamese_unet_s2             OK     epoch 11  val 0.3132  25.496 M  8.4s  
m2_siamese_segformer_ce_s0     OK     epoch  2  val 0.3624  14.836 M  12.6s  
m2_siamese_seg

## 4. M1 weighted-CE class weights vs the values proposed for M3 (exact equality, not rounding)

In [ ]:
# Class weights from the M3 proposal, the repo config, train.py defaults and each M1 CE checkpoint
CW = {"proposed_for_M3": M3_CLASS_WEIGHTS,
      "repo_config_m1_siamese_unet_ce.yaml": yaml.safe_load(open(f"{REPO}/configs/cv/m1_siamese_unet_ce.yaml"))["class_weights"],
      "train.py_DEFAULTS": __import__("floodsitrep.cv.train", fromlist=["DEFAULTS"]).DEFAULTS["class_weights"]}
for s in (0, 1, 2):
    run = f"m1_siamese_unet_ce_s{s}"
    c = DETAILS.get(run, {}).get("config")
    CW[f"{run}/best.pt"] = None if c is None else {"class_weights": c.get("class_weights"), "loss": c.get("loss")}
# All must be exactly equal (no rounding)
CW["all_identical"] = all(
    (v["class_weights"] if isinstance(v, dict) else v) == M3_CLASS_WEIGHTS for k, v in CW.items()
    if k != "proposed_for_M3" and v is not None)
CW["m1_ce_runs_all_use_ce"] = all(CW[f"m1_siamese_unet_ce_s{s}/best.pt"] and CW[f"m1_siamese_unet_ce_s{s}/best.pt"]["loss"] == "ce"
                                  for s in (0, 1, 2))
jdump(CW, out("inventory", "m3_class_weights_check.json"))
for k, v in CW.items():
    print(f"{k:40s} {v!r}")

proposed_for_M3                          [0.1, 0.52, 0.88, 0.75, 2.74]
repo_config_m1_siamese_unet_ce.yaml      [0.1, 0.52, 0.88, 0.75, 2.74]
train.py_DEFAULTS                        [0.1, 0.52, 0.88, 0.75, 2.74]
m1_siamese_unet_ce_s0/best.pt            {'class_weights': [0.1, 0.52, 0.88, 0.75, 2.74], 'loss': 'ce'}
m1_siamese_unet_ce_s1/best.pt            {'class_weights': [0.1, 0.52, 0.88, 0.75, 2.74], 'loss': 'ce'}
m1_siamese_unet_ce_s2/best.pt            {'class_weights': [0.1, 0.52, 0.88, 0.75, 2.74], 'loss': 'ce'}
all_identical                            True
m1_ce_runs_all_use_ce                    True


## 5. The saved Nepal results (read only)
Checks: `DONE` and `decision.txt` present; one `cmp_<run>_test.npz` per run; each covers exactly the 2,476 Nepal crops
in the same order as `make_dataset(..., sort=True)` (the order `building_level.run` predicts in); the headline numbers
recomputed from the saved confusion matrices with the pinned metric code match `nepal_comparison.csv` and the PDF;
the final model's comparison file equals `building_level_test.npz`.

In [ ]:
# Only the manifest is needed here (no images)
os.makedirs(X, exist_ok=True)
if not os.path.exists(MAN):
    shutil.copy(f"{D}/xbd/rscc_flood_manifest.csv", MAN)            # local copy; only the manifest is read here
from floodsitrep.cv.eval.infer import make_dataset
from floodsitrep.cv.eval import metrics as M
import floodsitrep.cv.eval.building_level as B
from floodsitrep.data.tiling import tile_name

# The Nepal crop order used by building_level.run
DS = make_dataset(X, MAN, "test", sort=True)           # reads the manifest only; no images
ORDER = [tile_name(r) for r in DS.rows]
print("Nepal crops in the manifest:", len(ORDER), "| images:", len({t.rsplit('_p', 1)[0] for t in ORDER}))

# What is in the saved Nepal folder
nep_files = sorted(os.listdir(NEPAL))
print("eval_extras/nepal_final:", nep_files)
print("DONE present:", "DONE" in nep_files)
print("--- decision.txt ---\n" + open(f"{NEPAL}/decision.txt").read())
dec_t = os.path.getmtime(f"{NEPAL}/decision.txt")
CMP_CSV = pd.read_csv(f"{NEPAL}/nepal_comparison.csv") if "nepal_comparison.csv" in nep_files else None

# For each run: check the saved file's crops and their order, and recompute its headline numbers
# from the saved confusion matrices
NEP = []
for run, (fam, seed) in EXPECTED.items():
    p = f"{NEPAL}/cmp_{run}_test.npz"
    r = {"run": run, "family": fam, "seed": seed, "file_present": os.path.exists(p)}
    if r["file_present"]:
        z = np.load(p)
        r["keys"] = " ".join(sorted(z.files))
        ids = [str(t) for t in z["crop_ids"]]
        r["crops"], r["images"] = len(ids), len(z["img_ids"])
        r["same_crops_same_order"] = ids == ORDER
        r["crop_cms_shape"] = str(z["crop_cms"].shape)
        r["buildings"] = int(z["bcms"].sum())
        r["written_after_decision_txt"] = os.path.getmtime(p) > dec_t
        px = M.flat(M.summarise(z["crop_cms"].sum(0)))
        b = B.summarise_buildings(z["bcms"].sum(0), z["comps"].sum(), z["falses"].sum())
        r.update({"pixel_score": px["xview2_pixel_score"], "loc_f1": px["loc_f1"], "minor": px["dmg_f1_minor"],
                  "major": px["dmg_f1_major"], "destroyed": px["dmg_f1_destroyed"], "miou5": px["miou"],
                  "building_score": b["xview2_building_score"], "bld_recall": b["loc_recall"]})
        if CMP_CSV is not None:
            row = CMP_CSV[(CMP_CSV["model"] == fam) & (CMP_CSV["seed"] == seed)]
            if len(row) == 1:
                row = row.iloc[0]
                r["max_abs_diff_vs_csv"] = max(abs(r[a] - row[b_]) for a, b_ in
                    [("pixel_score", "pixel score"), ("loc_f1", "loc F1"), ("minor", "minor"), ("major", "major"),
                     ("destroyed", "destroyed"), ("building_score", "building score"), ("bld_recall", "bld recall")])
            else:
                r["max_abs_diff_vs_csv"] = f"{len(row)} matching rows in csv"
    NEP.append(r)
NEP_DF = pd.DataFrame(NEP)
NEP_DF.to_csv(out("inventory", "nepal_saved_results_check.csv"), index=False)
print(NEP_DF.drop(columns=["keys"], errors="ignore").round(4).to_string(index=False))

# The final model's comparison file must equal its building-level test file
fm, fb = np.load(f"{NEPAL}/cmp_m1_siamese_unet_ce_s0_test.npz"), np.load(f"{NEPAL}/building_level_test.npz")
FINAL_SAME = all(np.array_equal(fm[k], fb[k]) for k in fb.files)
print("\nfinal model: cmp file identical to building_level_test.npz:", FINAL_SAME)
print("final model recomputed: pixel score %.4f (PDF 0.415), building score %.4f (PDF 0.443)" % (
    NEP_DF.loc[NEP_DF.run == "m1_siamese_unet_ce_s0", "pixel_score"].item(),
    NEP_DF.loc[NEP_DF.run == "m1_siamese_unet_ce_s0", "building_score"].item()))

Nepal crops in the manifest: 2476 | images: 619
eval_extras/nepal_final: ['building_level_test.npz', 'calibration_sample_test.npz', 'cmp_m0_early_fusion_unet_s0_test.npz', 'cmp_m0_early_fusion_unet_s1_test.npz', 'cmp_m0_early_fusion_unet_s2_test.npz', 'cmp_m1_siamese_unet_ce_s0_test.npz', 'cmp_m1_siamese_unet_ce_s1_test.npz', 'cmp_m1_siamese_unet_ce_s2_test.npz', 'cmp_m1_siamese_unet_s0_test.npz', 'cmp_m1_siamese_unet_s1_test.npz', 'cmp_m1_siamese_unet_s2_test.npz', 'cmp_m2_siamese_segformer_ce_s0_test.npz', 'cmp_m2_siamese_segformer_ce_s1_test.npz', 'cmp_m2_siamese_segformer_ce_s2_test.npz', 'decision.txt', 'nepal_comparison.csv', 'nepal_metrics.csv', 'nepal_metrics.json']
DONE present: False
--- decision.txt ---
Written 2026-11-18T13:29:49.601997 BEFORE any Nepal (test) result.
Final model: m1_siamese_unet_ce_s0/best.pt (epoch 12), temperature 1.133, both chosen on hurricane-florence (val).
Other models (all seeds of M1 CE, M1 Dice+focal, M0, M2) are scored on Nepal for comparison on

## 6. Clean-reproduction spot check (64 Nepal crops = 16 images, 4 checkpoints, clean images only)
Same function the Nepal notebook used (`building_level.run`), same defaults (batch 8, mixed precision on GPU, 2 workers),
same crop order, so the first 64 crops form the same 8 batches as in the saved run. Only the 64 files' images, targets
and label JSONs are copied (about 1 min). Predictions are compared **crop by crop** with the saved confusion
matrices, and building results image by image.

Expected: on a T4, identical matrices. On another GPU or the CPU, mixed-precision kernels can flip a few borderline
pixels; the cell reports how many. This result set the tolerance used for the full clean reproduction in the
corruption notebook.

In [ ]:
# One checkpoint from each family, seed 0
SPOT_RUNS = ["m1_siamese_unet_ce_s0", "m1_siamese_unet_s0", "m0_early_fusion_unet_s0", "m2_siamese_segformer_ce_s0"]
N_CROPS = 64
# Copy only the source files of the first 64 crops (16 images) to the local disk
need = {(r["split"], r["event"], r["tile_id"]) for r in DS.rows[:N_CROPS]}
t0 = time.time()
for split, event, tid in sorted(need):
    base = f"{event}_{tid}"
    for sub, fn in [("images", f"{base}_pre_disaster.png"), ("images", f"{base}_post_disaster.png"),
                    ("targets", f"{base}_post_disaster_target.png"), ("labels", f"{base}_post_disaster.json")]:
        dst = f"{X}/{split}/{sub}/{fn}"
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy(f"{D}/xbd/{split}/{sub}/{fn}", dst)      # read from Drive, write to local disk only
print(f"copied files for {len(need)} images in {time.time() - t0:.0f}s")

# Re-predict the 64 crops and compare crop by crop with the saved results
SPOT = []
for run in SPOT_RUNS:
    t0 = time.time()
    new = B.run(f"{CK}/{run}/best.pt", X, MAN, role="test", limit=N_CROPS,
                out_path=out("inventory", "clean_spotcheck", f"{run}_first{N_CROPS}.npz"))
    secs = time.time() - t0
    old = np.load(f"{NEPAL}/cmp_{run}_test.npz")
    oi = {str(t): i for i, t in enumerate(old["crop_ids"])}
    same, moved = 0, 0
    for i, t in enumerate(new["crop_ids"]):
        a, b = new["crop_cms"][i], old["crop_cms"][oi[str(t)]]
        same += bool(np.array_equal(a, b))
        moved += int(np.abs(a - b).sum()) // 2               # about the number of pixels whose prediction changed
    # Building-level results compared image by image
    oj = {str(t): i for i, t in enumerate(old["img_ids"])}
    img_same = sum(bool(np.array_equal(new["bcms"][i], old["bcms"][oj[str(t)]])
                        and new["comps"][i] == old["comps"][oj[str(t)]] and new["falses"][i] == old["falses"][oj[str(t)]])
                   for i, t in enumerate(new["img_ids"]))
    sel = [oi[str(t)] for t in new["crop_ids"]]
    s_new = M.flat(M.summarise(new["crop_cms"].sum(0)))["xview2_pixel_score"]
    s_old = M.flat(M.summarise(old["crop_cms"][sel].sum(0)))["xview2_pixel_score"]
    px_total = int(new["crop_cms"].sum())
    SPOT.append({"run": run, "device": ENV["gpu"], "crops": len(new["crop_ids"]), "crops_identical": same,
                 "images": len(new["img_ids"]), "images_identical_building_results": img_same,
                 "pixels_changed_approx": moved, "valid_pixels": px_total, "share_changed": moved / max(px_total, 1),
                 "score_64_new": s_new, "score_64_saved": s_old, "score_abs_diff": abs(s_new - s_old),
                 "seconds": round(secs, 1)})
    print(SPOT[-1], flush=True)
# Verdict: exact only if every crop and every image is identical
SPOT_DF = pd.DataFrame(SPOT)
SPOT_DF.to_csv(out("inventory", "clean_spotcheck", "spotcheck_summary.csv"), index=False)
exact = bool((SPOT_DF.crops_identical == SPOT_DF.crops).all() and (SPOT_DF.images_identical_building_results == SPOT_DF.images).all())
SPOT_VERDICT = ("EXACT: crop-level and building-level results identical to the saved Nepal run" if exact else
                f"NOT EXACT: max share of pixels changed {SPOT_DF.share_changed.max():.2e}, "
                f"max score difference {SPOT_DF.score_abs_diff.max():.4f} (see table)")
print("\n" + SPOT_VERDICT)

copied files for 16 images in 44s
{'run': 'm1_siamese_unet_ce_s0', 'device': 'Tesla T4', 'crops': 64, 'crops_identical': 38, 'images': 16, 'images_identical_building_results': 16, 'pixels_changed_approx': 61, 'valid_pixels': 15237884, 'share_changed': 4.003180494089599e-06, 'score_64_new': 0.28150999570555785, 'score_64_saved': 0.28145633519214436, 'score_abs_diff': 5.366051341348932e-05, 'seconds': 8.1}
{'run': 'm1_siamese_unet_s0', 'device': 'Tesla T4', 'crops': 64, 'crops_identical': 50, 'images': 16, 'images_identical_building_results': 16, 'pixels_changed_approx': 23, 'valid_pixels': 15237884, 'share_changed': 1.5093959240009965e-06, 'score_64_new': 0.29090197885987135, 'score_64_saved': 0.29089536684788386, 'score_abs_diff': 6.612011987483513e-06, 'seconds': 7.1}
{'run': 'm0_early_fusion_unet_s0', 'device': 'Tesla T4', 'crops': 64, 'crops_identical': 46, 'images': 16, 'images_identical_building_results': 15, 'pixels_changed_approx': 35, 'valid_pixels': 15237884, 'share_changed': 

## 7. Preservation check and summary

In [ ]:
# Compare with the snapshot from section 0: nothing outside m3_experiments/ may have changed
AFTER = snapshot()
added = sorted(set(AFTER) - set(BEFORE))
removed = sorted(set(BEFORE) - set(AFTER))
changed = sorted(k for k in set(BEFORE) & set(AFTER) if BEFORE[k] != AFTER[k])
PRES = {"entries_checked": len(BEFORE), "added_outside_m3_experiments": added,
        "removed_outside_m3_experiments": removed, "changed_outside_m3_experiments": changed,
        "verdict": "NOTHING OUTSIDE m3_experiments CHANGED" if not (added or removed or changed) else
                   "DIFFERENCES FOUND (a teammate may be working in the folder at the same time; see lists)"}
jdump(PRES, out("inventory", "preservation_check.json"))

# Summary of every check, saved as JSON and printed
inv_cols = ["run", "status", "best_epoch", "ckpt_val_score", "params_M", "best_pt_MB", "issues"]
SUMMARY = {
    "when": datetime.datetime.now().isoformat(timespec="minutes"),
    "write_test": WT["verdict"],
    "code": {"pinned": CODE_EQ["pinned_commit"], "identical_to_nepal_run_branch": CODE_EQ["identical"],
             "files_differing": CODE_EQ["files_differing"]},
    "environment": ENV,
    "checkpoints": INV_DF[[c for c in inv_cols if c in INV_DF]].to_dict(orient="records"),
    "class_weights_identical": CW["all_identical"], "m1_ce_loss_is_ce": CW["m1_ce_runs_all_use_ce"],
    "nepal_saved": {"DONE": "DONE" in nep_files, "cmp_files_present": int(NEP_DF.file_present.sum()),
                    "all_same_crops_same_order": bool(NEP_DF.get("same_crops_same_order", pd.Series([False])).fillna(False).all()),
                    "final_cmp_equals_building_level_test": FINAL_SAME,
                    "max_abs_diff_vs_csv": NEP_DF.get("max_abs_diff_vs_csv", pd.Series(dtype=object)).tolist()},
    "spot_check": {"verdict": SPOT_VERDICT, "rows": SPOT},
    "preservation": PRES["verdict"],
}
jdump(SUMMARY, out("inventory", "stage2_summary.json"))
print(json.dumps(SUMMARY, indent=1, default=str))
# List every file this notebook wrote
print("\nFiles written (all inside m3_experiments/):")
for dp, _, fs in os.walk(EXP):
    for f in fs:
        print("  ", os.path.relpath(os.path.join(dp, f), D))

{
 "when": "2026-11-20T12:02",
 "write_test": "WRITE ACCESS OK",
 "code": {
  "pinned": "8f10e61 2026-11-20T14:07:06+04:00 Merge pull request #28 from aaarrvind/m5/generate-advisory",
  "identical_to_nepal_run_branch": true,
  "files_differing": []
 },
 "environment": {
  "torch": "2.14.1+cu130",
  "torchvision": "0.29.1+cu130",
  "transformers": "5.22.1",
  "numpy": "2.4.6",
  "scipy": "1.16.3",
  "pillow": "11.3.0",
  "pandas": "2.2.3",
  "python": "3.13.16",
  "gpu": "Tesla T4"
 },
 "checkpoints": [
  {
   "run": "m0_early_fusion_unet_s0",
   "status": "OK",
   "best_epoch": 9,
   "ckpt_val_score": 0.2858378085984318,
   "params_M": 24.446,
   "best_pt_MB": 98.0,
   "issues": ""
  },
  {
   "run": "m0_early_fusion_unet_s1",
   "status": "OK",
   "best_epoch": 7,
   "ckpt_val_score": 0.2737612939752898,
   "params_M": 24.446,
   "best_pt_MB": 98.0,
   "issues": ""
  },
  {
   "run": "m0_early_fusion_unet_s2",
   "status": "OK",
   "best_epoch": 8,
   "ckpt_val_score": 0.2676778470338

## End of notebook
Releases the Colab runtime so it stops using compute units.

In [ ]:
# Release the Colab runtime
from google.colab import runtime
runtime.unassign()